In [ ]:
# Import necessary libraries.
import pandas as pd
import yaml
import math

In [ ]:
# Helper function to map numbers to letters of the alphabet.
def int_to_letter(n):
    if n < 0 or n > 25:
        raise ValueError("Input must be an integer between 0 and 25 inclusive.")
    return chr(n + ord('a'))

In [ ]:
# Step 1: Load the data from an XLSX file.
def load_data(file_path):
    """
    Load data from an XLSX file.
    
    Args:
        file_path (str): The path to the input file.
        
    Returns:
        pd.DataFrame: The loaded data as a DataFrame.
    """
    if file_path.endswith('.xlsx') or file_path.endswith('.xls'):
        return pd.read_excel(file_path, sheet_name=['L1 UC', 'L2 UC', 'L3 UC'])
    else:
        raise ValueError("Unsupported file format. Please provide a CSV or XLSX file.")

In [ ]:
# Helper functions to handle nan values.
def replace_nan_with_zero(value):
    return value if not math.isnan(value) else 0

# Helper function to handle nan values.
def replace_nan_with_empty_string(value):
    return value if not str(value) == 'nan' else ''

In [ ]:
# Process the L1 data to convert it into a list of dictionaries, representing the desired YAML format.
def process_data_L1(df, domain):
    """
    Process the DataFrame to create a list of dictionaries in the desired YAML format.
    
    Args:
        df (pd.DataFrame): The input DataFrame.
        domain (str): the ATT&CK domain that all use cases are assigned to.
        
    Returns:
        list: A list of dictionaries representing the YAML usecase entries.
    """

    usecases = []

    for index, row in df.iterrows():
        if index > 5:  # Skip the first five rows
            name = str(row['Unnamed: 3'])  # This name is messy, but it points to the correct column.
            id = str(row['Strategic Overview'])  # This name is messy, but it points to the correct column.
            description = replace_nan_with_empty_string(row['Unnamed: 17'])  # This name is messy, but it points to the correct column.
            if id == 'nan':  # Stop processing when we get 'nan', which occurs for rows without a L1 UC id. Here, we assume that the list of use cases has ended.
                break

            # Process all rows. Create a usecase for each row.
            usecase = {
                'domain': domain,
                'level': 1,
                'name': name,
                'id': id,
                'description': description,
            }

            usecases.append(usecase)

    return usecases

In [ ]:
# Process the L2 data to convert it into a list of dictionaries, representing the desired YAML format.
def process_data_L2(df, domain):
    """
    Process the DataFrame to create a list of dictionaries in the desired YAML format.
    
    Args:
        df (pd.DataFrame): The input DataFrame.
        domain (str): the ATT&CK domain that all use cases are assigned to.
        
    Returns:
        list: A list of dictionaries representing the YAML usecase entries.
    """

    usecases = []

    for index, row in df.iterrows():
        name = str(row['Use Case Name'])
        id = row['L2 Use Case Identifier']
        description = replace_nan_with_empty_string(row['Use Case Description'])
        parents = row['L1 Use Case Identifier']

        # Pre-process parents data.
        if pd.isnull(parents):
            parents = []
        else:
            parents = parents.split(',')

        # Process all rows. Create a usecase for each row.
        usecase = {
            'domain': domain,
            'level': 2,
            'name': name,
            'id': id,
            'description': description,
            'parentIds': parents,
        }

        usecases.append(usecase)

    return usecases

In [ ]:
# Process the L3 data to convert it into a list of dictionaries, representing the desired YAML format.
def process_data_L3(df, domain):
    """
    Process the DataFrame to create a list of dictionaries in the desired YAML format.
    
    Args:
        df (pd.DataFrame): The input DataFrame.
        domain (str): the ATT&CK domain that all use cases are assigned to.
        
    Returns:
        list: A list of dictionaries representing the YAML usecase entries.
    """

    usecases = []

    for index, row in df.iterrows():
        name_base = str(row['Technical Use Case Name'])
        id_base = row['Rule Identifier']
        description = replace_nan_with_empty_string(row['Use Case Description'])
        implementation = round(replace_nan_with_zero(row['Implementation %']) * 100, 2)
        effectiveness = round(replace_nan_with_zero(row['Effectiveness %']) * 100, 2)
        visibility = round(replace_nan_with_zero(row['Visibility %']) * 100, 2)
        override = visibility != 0  # Set override to True when visibility is not 0 (otherwise set to False).
        parents = row['L2 Use Case Identifier']
        techniques = row['ATT&CK techniques']

        # Pre-process techniques data.
        if pd.isnull(techniques):
            techniques = []
        else:
            techniques = techniques.split(',')

        # Pre-process parents data.
        if pd.isnull(parents):
            parents = []
        else:
            parents = parents.split(',')

        # Process all rows. Create a usecase for each row, or multiple usecases when multiple techniques are present.
        usecase_has_multiple_techniques = len(techniques) > 1
        for i, technique in enumerate(techniques):

            # Create a name and id for the use case, based on whether the use case has multiple techniques and has to be split up.
            if usecase_has_multiple_techniques:
                name = name_base + ' - ' + int_to_letter(i).upper()
                id = id_base + int_to_letter(i).upper()
            else:
                name = name_base
                id = id_base
            
            usecase = {
                'domain': domain,
                'level': 3,
                'name': name,
                'id': id,
                'description': description,
                'parentIds': parents,
                'visibility': visibility,
                'implementation': implementation,
                'effectiveness': effectiveness,
                'weight': 0,
                'impact': 0,
                'attackTechniqueId': technique,
                'visibilityFromAttackTechniqueOverride': override
            }

            usecases.append(usecase)

    return usecases


In [ ]:
# Step 3: Save the processed data to a YAML file.
def save_to_yaml(yaml_entries, output_file):
    """
    Save the processed data to a YAML file.
    
    Args:
        yaml_entries (list): The list of dictionaries representing the YAML entries.
        output_file (str): The path to the output YAML file.
    """
    with open(output_file, 'w') as yaml_file:
        yaml.dump(yaml_entries, yaml_file, default_flow_style=False)


In [ ]:
# Step 4: Main function to execute the conversion.
def main(input_file, output_file, domain):
    """
    Main function to load data, process it, and save the result to a YAML file.
    
    Args:
        input_file (str): The path to the input CSV or XLSX file.
        output_file (str): The path to the output YAML file.
    """
    # Load data from the input file.
    df = load_data(input_file)
    
    # Process the DataFrame to create a list of dictionaries, representing the usecases.
    L1_usecases = process_data_L1(df['L1 UC'], domain)
    L2_usecases = process_data_L2(df['L2 UC'], domain)
    L3_usecases = process_data_L3(df['L3 UC'], domain)
    
    # Save the YAML entries to a file.
    save_to_yaml(L1_usecases + L2_usecases + L3_usecases, output_file)
    
    print(f"Data successfully converted and saved to {output_file}")


In [ ]:
# Example usage.
input_file = '../example_data_dettect/magma_export_conversion_test/example_export_magma_file.xlsx'  # Change this to your input file path.
output_file = '../example_data_dettect/magma_export_conversion_test/magma_file_magma_conversion_output.yaml'  # Change this to your desired output YAML file path.
domain = 'enterprise-attack'

main(input_file, output_file, domain)